# Synthetic Data Generator
### Step 7 : Kafka Producer Adapter

In [ ]:
from utils.postgres_util import (
    get_engine_from_env, 
    read_sql_dataframe,
)


from utils.synthetic.pipeline.kafka_producer_adapter import (
    run_send_queue_producer_once, 
    run_send_queue_producer_loop, 
    build_sensor_message_payload, 
    json_dumps_safe,
)

In [2]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

IS_ENABLED_FLAG = True

PRODUCER_TOPIC = str("pump.telemetry.synthetic")
PRODUCER_BATCH_SIZE = int(5200)
PRODUCER_POLL_SECONDS = float(0.0)
PRODUCER_MAX_SEND_ATTEMPTS = int(3)

PRODUCER_WORKER_ID = str("producer_worker_001")
CLIENT_ID = str("pump-telemetry-producer")

SIMULATION_TABLE_NAME = str("simulation_state_control")
SEND_QUEUE_TABLE_NAME = str("synthetic_sensor_messages_send_queue")

FLUSH_TIMEOUT_SECONDS = float(30.0)

MAX_BATCHES = int(10)
STOP_ON_FAILURE_FLAG = True


---

In [4]:


engine = get_engine_from_env()


---

#### One Batch

In [ ]:

result = run_send_queue_producer_once(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    schema=SCHEMA,
    queue_table=SEND_QUEUE_TABLE_NAME,
    control_table=SIMULATION_TABLE_NAME,
    producer_worker_id=PRODUCER_WORKER_ID,
    client_id=CLIENT_ID,
    flush_timeout_seconds=FLUSH_TIMEOUT_SECONDS,
)

display(result)

----

#### Loop

In [ ]:
results = run_send_queue_producer_loop(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    schema=SCHEMA,
    queue_table=SEND_QUEUE_TABLE_NAME,
    control_table=SIMULATION_TABLE_NAME,
    producer_worker_id=PRODUCER_WORKER_ID,
    client_id=CLIENT_ID,
    max_batches=MAX_BATCHES,
    stop_on_failure=STOP_ON_FAILURE_FLAG,
    flush_timeout_seconds=FLUSH_TIMEOUT_SECONDS,
)

display(results)

----


#### Preview Payload Before Sending

In [ ]:
preview_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT *
    FROM capstone.synthetic_sensor_messages_send_queue
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 1
    """
)

payload = build_sensor_message_payload(preview_dataframe.iloc[0].to_dict())
print(json_dumps_safe(payload))

{"message_key": "pump_asset_001|1|41", "dataset_id": "pump_synthetic_v1", "run_id": "premelt_run_001", "asset_id": "pump_asset_001", "generated_row_id": "premelt_run_001_obs_000000000001", "observation": {"observation_index": 1, "observation_timestamp": "2026-03-19T08:00:00+00:00", "batch_id": 1, "row_in_batch": 0, "global_cycle_id": 1, "stream_state": "normal", "phase": "normal"}, "sensor": {"sensor_name": "sensor_41", "sensor_index": 41, "sensor_value": 36.76771926879883, "message_sequence_index": 0}, "metadata": {"meta_episode_id": 0, "meta_primary_fault_type": "step_shift", "meta_magnitude": 1.823247399889228, "created_at": "2026-03-20T01:34:26.977210+00:00"}, "telemetry": {"is_telemetry_event": false, "telemetry_event_type": null}, "producer": {"producer_send_attempt": 1, "queued_at": "2026-03-25T03:36:21.059875+00:00"}}


----

In [ ]:
# Dispose Engine
engine.dispose()